# Strain-Aware MIC Prediction — Clean Ablation Pipeline

Predict peptide **minimum inhibitory concentration (pMIC)** against specific
bacterial targets from a frozen **ESM-2** peptide embedding, and test whether
adding *pathogen-side* information helps.

**Everything heavy is cached on Google Drive** (`PROJECT_DIR`): the dataset,
the cluster-disjoint split, the ESM-2 peptide embeddings, and the PubMedBERT
species-description embeddings are loaded if present and only computed when
missing.

### Study design
A single shared MLP head (Huber loss, fixed seed, same split) is trained under
five feature regimes:

1. **Peptide-only** — ESM-2 embedding alone.
2. **Peptide + Strain ID** — + a learned embedding of the exact strain label.
3. **Peptide + Species Description** — + PubMedBERT embedding of a short
   biological description of the target species.
4. **Peptide + Taxonomy** — + categorical Gram stain / genus / family.
5. **Shuffled-Description Control** — variant 3 with species→embedding mapping
   permuted (sanity check: real description signal should beat this).

Each regime is evaluated under three **separate** protocols — standard
clustered split, zero-shot (held-out targets), few-shot (sparse targets) — plus
a per-target long-tail analysis.


## 1 — Setup: Drive, paths, device, caching helper

In [ ]:
# Mount Google Drive, pin a single PROJECT_DIR, and define one cache helper used
# everywhere below: compute-only-if-missing.
import os, json, random
from pathlib import Path
import numpy as np
import torch

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/dbaasp_backup")
except Exception:
    # Local fallback so the notebook is runnable off-Colab.
    PROJECT_DIR = Path(os.environ.get("PROJECT_DIR", "./dbaasp_backup"))

PHASE1_DIR = PROJECT_DIR / "phase1_output"
PHASE2_DIR = PROJECT_DIR / "phase2_output"
CACHE_DIR  = PROJECT_DIR / "ablation_cache"       # new artifacts from this notebook
CACHE_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed()

# Column names (fixed by the upstream dataset).
TARGET  = "pMIC_point"                     # regression target (higher = more potent)
SEQ     = "sequence_raw"                    # peptide sequence
SPECIES = "target_species_standardized"     # species-level target label
STRAIN  = "target_normalized"               # strain-level (exact) target label
CLUSTER = "cluster_id"                       # 80%-identity CD-HIT cluster

print("PROJECT_DIR:", PROJECT_DIR)
print("Device     :", DEVICE, "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# Dependencies for the two encoders (no-op if already present; only the model
# loads below actually need them, and those run only when a cache is missing).
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fair-esm", "transformers"], check=False)

## 2 — Load cached dataset & cluster-disjoint split

The upstream Phase-1/Phase-2 steps (DBAASP collection, MIC normalisation,
CD-HIT 80%-identity clustering, 75/15/10 cluster-aware split) are already cached
on Drive. We simply load the three split CSVs; each row carries the peptide, its
pMIC, the strain/species labels, and its cluster id.

In [ ]:
import pandas as pd

def load_split(name):
    p = PHASE2_DIR / f"phase2_{name}.csv"
    if not p.exists():
        raise FileNotFoundError(
            f"Expected cached split {p}. Run the upstream Phase-2 split once; "
            "this notebook consumes its cached output.")
    df = pd.read_csv(p)
    df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
    df = df.dropna(subset=[SEQ, TARGET, STRAIN, CLUSTER]).copy()
    df[SEQ] = df[SEQ].astype(str).str.strip().str.upper()
    for c in (SPECIES, STRAIN):
        df[c] = df[c].fillna("unknown").astype(str).str.strip()
    return df.reset_index(drop=True)

train_df = load_split("train")
val_df   = load_split("validation")
test_df  = load_split("test")

for n, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{n:5s}: {len(d):6,d} rows | {d[STRAIN].nunique():4d} strains | {d[SPECIES].nunique():3d} species")
print("Target pMIC range:", round(train_df[TARGET].min(), 2), "to", round(train_df[TARGET].max(), 2))

## 3 — ESM-2 peptide embeddings (cached)

One mean-pooled ESM-2 embedding per unique peptide. The backbone is a single
switch — `ESM_BACKBONE = "8M"` (320-d, fast) or `"150M"` (640-d, stronger) — and
each backbone caches to its own `.npz`, so you can keep both and A/B them. The
model is only loaded and run when embeddings are missing.

In [ ]:
# --- ESM-2 backbone switch: flip ESM_BACKBONE to swap the model everywhere ---
ESM_BACKBONE = "8M"                      # "8M" (fast, 320-d) or "150M" (stronger, 640-d)
ESM_REGISTRY = {
    "8M":   {"loader": "esm2_t6_8M_UR50D",    "layer": 6,  "dim": 320},
    "150M": {"loader": "esm2_t30_150M_UR50D", "layer": 30, "dim": 640},
}
_cfg = ESM_REGISTRY[ESM_BACKBONE]
ESM_LAYER, ESM_DIM = _cfg["layer"], _cfg["dim"]
ESM_COLS = [f"ESM_{i:03d}" for i in range(ESM_DIM)]
ESM_NPZ = CACHE_DIR / f"esm2_{ESM_BACKBONE}_peptide_emb.npz"   # per-backbone cache (no collision)
print(f"ESM-2 backbone: {ESM_BACKBONE}  (layer {ESM_LAYER}, dim {ESM_DIM})  cache: {ESM_NPZ.name}")

all_seqs = sorted(set(train_df[SEQ]) | set(val_df[SEQ]) | set(test_df[SEQ]))

def _save_esm(lookup):
    # Atomic checkpoint so an interrupted/disconnected run resumes, not restarts.
    ks = list(lookup)
    tmp = str(ESM_NPZ) + ".tmp"
    np.savez_compressed(tmp, sequences=np.array(ks), embeddings=np.stack([lookup[k] for k in ks]))
    os.replace(tmp, ESM_NPZ)

def compute_esm_embeddings(seqs, lookup, checkpoint_every=20):
    import esm
    from tqdm.auto import tqdm
    model, alphabet = getattr(esm.pretrained, _cfg["loader"])()
    model = model.eval().to(DEVICE)
    bc = alphabet.get_batch_converter()
    B = 16
    for bi, i in enumerate(tqdm(range(0, len(seqs), B), desc=f"ESM-2 {ESM_BACKBONE}")):
        batch = [(f"s{j}", s) for j, s in enumerate(seqs[i:i+B])]
        _, _, toks = bc(batch)
        toks = toks.to(DEVICE)
        lengths = (toks != alphabet.padding_idx).sum(1)
        with torch.no_grad():
            rep = model(toks, repr_layers=[ESM_LAYER])["representations"][ESM_LAYER]
        for j, L in enumerate(lengths):
            lookup[batch[j][1]] = rep[j, 1:L - 1].mean(0).cpu().numpy().astype(np.float32)
        if (bi + 1) % checkpoint_every == 0:
            _save_esm(lookup)       # periodic checkpoint to Drive
    _save_esm(lookup)
    return lookup

if ESM_NPZ.exists():
    z = np.load(ESM_NPZ, allow_pickle=True)
    esm_lookup = dict(zip(z["sequences"].astype(str), z["embeddings"].astype(np.float32)))
    z.close()
    missing = [s for s in all_seqs if s not in esm_lookup]
    print(f"Loaded {len(esm_lookup):,} cached ESM-2 embeddings; {len(missing)} missing.")
else:
    esm_lookup, missing = {}, all_seqs
    print(f"No ESM-2 cache; computing {len(missing):,} embeddings.")

if missing:
    # Resume-safe: checkpoints into ESM_NPZ as it goes, so a dropped kernel
    # continues from the last checkpoint on rerun instead of recomputing all.
    compute_esm_embeddings(missing, esm_lookup)
    print(f"Saved {len(esm_lookup):,} ESM-2 embeddings -> {ESM_NPZ}")

def esm_matrix(df):
    return np.stack([esm_lookup[s] for s in df[SEQ]]).astype(np.float32)

## 4 — Pathogen features (species-level)

For every unique target species we build a **2-sentence biological description**
(Gram stain, cell wall, resistance), encode it once with **PubMedBERT**
(mean-pooled, `torch.no_grad`), and cache to `species_pubmedbert_emb.pt`. We also
attach simple categorical **taxonomy** (`gram_stain`, `genus`, `family`) — no
GNNs.

In [ ]:
# Curated biology for the dominant pathogens; everything else gets a templated
# 2-sentence description and genus parsed from the name.
CURATED = {
    "escherichia coli":       ("gram-negative", "escherichia",  "enterobacteriaceae",
        "Escherichia coli is a Gram-negative enteric bacterium with a thin peptidoglycan layer and an outer membrane rich in lipopolysaccharide. Its outer membrane and active efflux pumps limit entry of many antibiotics and cationic antimicrobial peptides."),
    "pseudomonas aeruginosa": ("gram-negative", "pseudomonas",  "pseudomonadaceae",
        "Pseudomonas aeruginosa is a Gram-negative opportunistic pathogen with a low-permeability outer membrane and powerful multidrug efflux systems. It is intrinsically resistant to many antibiotics and readily adapts its lipopolysaccharide to resist membrane-active peptides."),
    "staphylococcus aureus":  ("gram-positive", "staphylococcus","staphylococcaceae",
        "Staphylococcus aureus is a Gram-positive coccus with a thick peptidoglycan cell wall and no outer membrane. Resistance arises through methicillin resistance (mecA) and cell-surface modifications that reduce binding of cationic antimicrobial peptides."),
    "klebsiella pneumoniae":  ("gram-negative", "klebsiella",   "enterobacteriaceae",
        "Klebsiella pneumoniae is a Gram-negative enteric bacterium encased in a thick polysaccharide capsule over an LPS outer membrane. The capsule and carbapenemase enzymes make it a major multidrug-resistant hospital pathogen."),
    "acinetobacter baumannii":("gram-negative", "acinetobacter","moraxellaceae",
        "Acinetobacter baumannii is a Gram-negative coccobacillus with a highly impermeable outer membrane and broad intrinsic resistance. It survives desiccation and accumulates resistance determinants, including carbapenemases, in healthcare settings."),
    "enterococcus faecalis":  ("gram-positive", "enterococcus", "enterococcaceae",
        "Enterococcus faecalis is a Gram-positive gut commensal with a thick peptidoglycan wall and no outer membrane. It is intrinsically tolerant to many antibiotics and can acquire vancomycin resistance."),
    "enterococcus faecium":   ("gram-positive", "enterococcus", "enterococcaceae",
        "Enterococcus faecium is a Gram-positive coccus with a thick cell wall and no outer membrane. It is a leading vancomycin-resistant (VRE) nosocomial pathogen."),
    "enterobacter cloacae":   ("gram-negative", "enterobacter", "enterobacteriaceae",
        "Enterobacter cloacae is a Gram-negative enteric bacterium with an LPS outer membrane and inducible AmpC beta-lactamases. These enzymes and efflux systems drive broad beta-lactam resistance."),
    "bacillus subtilis":      ("gram-positive", "bacillus",     "bacillaceae",
        "Bacillus subtilis is a Gram-positive, spore-forming soil bacterium with a thick peptidoglycan wall and no outer membrane. It is generally antibiotic-susceptible and widely used as a model Gram-positive target."),
    "salmonella enterica":    ("gram-negative", "salmonella",   "enterobacteriaceae",
        "Salmonella enterica is a Gram-negative enteric pathogen with an LPS-rich outer membrane. PhoPQ-regulated lipid A modifications reduce susceptibility to cationic antimicrobial peptides."),
    "listeria monocytogenes": ("gram-positive", "listeria",     "listeriaceae",
        "Listeria monocytogenes is a Gram-positive intracellular pathogen with a thick peptidoglycan wall and no outer membrane. Surface charge modifications contribute to resistance against antimicrobial peptides."),
    "micrococcus luteus":     ("gram-positive", "micrococcus",  "micrococcaceae",
        "Micrococcus luteus is a Gram-positive coccus with a thick peptidoglycan wall and no outer membrane. It is generally susceptible and often used as a sensitive assay strain."),
    "candida albicans":       ("fungus",        "candida",      "debaryomycetaceae",
        "Candida albicans is a fungal pathogen with a chitin and beta-glucan cell wall surrounded by a mannoprotein layer rather than a bacterial envelope. Its membrane ergosterol and efflux pumps influence antifungal and peptide susceptibility."),
}

def describe_species(name):
    key = str(name).strip().lower()
    for k, (gram, genus, family, desc) in CURATED.items():
        if k in key:
            return gram, genus, family, desc
    genus = key.split()[0] if key and key != "unknown" else "unknown"
    desc = (f"{name} is a microbial target used in antimicrobial-peptide susceptibility "
            f"testing, and its cell envelope governs peptide entry. Its intrinsic "
            f"resistance mechanisms determine sensitivity to membrane-active peptides.")
    return "unknown", genus, "unknown", desc

species_list = sorted(set(train_df[SPECIES]) | set(val_df[SPECIES]) | set(test_df[SPECIES]))
species_meta = {s: describe_species(s) for s in species_list}
taxonomy_df = pd.DataFrame(
    [(s, g, gn, fm) for s, (g, gn, fm, _) in species_meta.items()],
    columns=[SPECIES, "gram_stain", "genus", "family"])
print(f"{len(species_list)} unique species described. Sample:")
display(taxonomy_df.head(8))
# If most species fall through to the generic template, the description/taxonomy
# variants are being under-sold; curate more species if coverage is low.
print("Curated coverage (species with known Gram stain): "
      f"{(taxonomy_df['gram_stain'] != 'unknown').mean():.1%} of species, "
      f"{train_df[SPECIES].isin(taxonomy_df.loc[taxonomy_df['gram_stain'] != 'unknown', SPECIES]).mean():.1%} of train rows")

In [ ]:
# PubMedBERT species-description embeddings (768-d, mean-pooled), cached to Drive.
PUBMEDBERT_PT = CACHE_DIR / "species_pubmedbert_emb.pt"
PUBMEDBERT_DIM = 768
MODEL_NAME = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"

def compute_pubmedbert(species_to_desc):
    from transformers import AutoTokenizer, AutoModel
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    enc = AutoModel.from_pretrained(MODEL_NAME).eval().to(DEVICE)
    names = list(species_to_desc)
    vecs = {}
    B = 16
    for i in range(0, len(names), B):
        chunk = names[i:i+B]
        t = tok([species_to_desc[n] for n in chunk], padding=True, truncation=True,
                max_length=128, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = enc(**t).last_hidden_state                 # (B, L, 768)
        mask = t["attention_mask"].unsqueeze(-1).float()
        pooled = (out * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        for j, n in enumerate(chunk):
            vecs[n] = pooled[j].cpu().numpy().astype(np.float32)
    return vecs

desc_map = {s: species_meta[s][3] for s in species_list}
if PUBMEDBERT_PT.exists():
    blob = torch.load(PUBMEDBERT_PT, weights_only=False)   # dict of numpy arrays
    species_emb = {k: np.asarray(v, np.float32) for k, v in blob["emb"].items()}
    new = [s for s in species_list if s not in species_emb]
    print(f"Loaded {len(species_emb)} cached PubMedBERT species embeddings; {len(new)} new.")
    if new:
        species_emb.update(compute_pubmedbert({s: desc_map[s] for s in new}))
        torch.save({"emb": species_emb, "desc": desc_map}, PUBMEDBERT_PT)
else:
    print(f"No PubMedBERT cache; encoding {len(species_list)} descriptions.")
    species_emb = compute_pubmedbert(desc_map)
    torch.save({"emb": species_emb, "desc": desc_map}, PUBMEDBERT_PT)
    print(f"Saved -> {PUBMEDBERT_PT}")

_zero768 = np.zeros(PUBMEDBERT_DIM, np.float32)
def species_emb_matrix(df, mapping=None):
    mapping = mapping or {}
    return np.stack([species_emb.get(mapping.get(s, s), _zero768) for s in df[SPECIES]]).astype(np.float32)

# Shuffled-description control: permute species -> embedding (fixed seed).
rng = np.random.default_rng(SEED)
_perm = list(species_list); rng.shuffle(_perm)
SHUFFLE_MAP = dict(zip(species_list, _perm))
print("Shuffled-control mapping built (permuted species descriptions).")

## 5 — Shared MLP head, encoders, training & metrics

One regression head (two hidden layers, LayerNorm, GELU, dropout) is reused for
all five variants. Conditioning differs only in the input block that is
concatenated to the ESM-2 vector; the Strain-ID variant adds a learned
`nn.Embedding`. Loss = **Huber (SmoothL1)**; optimiser, schedule, seed and split
are identical across variants.

In [ ]:
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import OneHotEncoder
from scipy.stats import pearsonr, spearmanr, kendalltau

HID, DROPOUT, EPOCHS, PATIENCE, BATCH = 128, 0.2, 60, 10, 128

class AblationMLP(nn.Module):
    def __init__(self, esm_dim=ESM_DIM, cond_dim=0, n_strains=0, strain_dim=32):
        super().__init__()
        self.use_strain = n_strains > 0
        in_dim = esm_dim + cond_dim + (strain_dim if self.use_strain else 0)
        if self.use_strain:
            self.strain_emb = nn.Embedding(n_strains, strain_dim)
        self.head = nn.Sequential(
            nn.Linear(in_dim, HID), nn.LayerNorm(HID), nn.GELU(), nn.Dropout(DROPOUT),
            nn.Linear(HID, HID // 2), nn.LayerNorm(HID // 2), nn.GELU(), nn.Dropout(DROPOUT),
            nn.Linear(HID // 2, 1))
    def forward(self, x_esm, x_cond, strain_idx):
        parts = [x_esm]
        if x_cond.shape[1] > 0:
            parts.append(x_cond)
        if self.use_strain:
            parts.append(self.strain_emb(strain_idx))
        return self.head(torch.cat(parts, dim=-1))

def fit_encoders(tr):
    # Strain vocab (idx 0 = unknown/unseen) and taxonomy one-hot, fit on train only.
    strains = ["<unk>"] + sorted(tr[STRAIN].unique())
    strain2idx = {s: i for i, s in enumerate(strains)}
    tax = tr[[SPECIES]].merge(taxonomy_df, on=SPECIES, how="left")[["gram_stain", "genus", "family"]].fillna("unknown")
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    ohe.fit(tax)
    return {"strain2idx": strain2idx, "ohe": ohe}

def taxonomy_matrix(df, enc):
    tax = df[[SPECIES]].merge(taxonomy_df, on=SPECIES, how="left")[["gram_stain", "genus", "family"]].fillna("unknown")
    return enc["ohe"].transform(tax).astype(np.float32)

def strain_idx(df, enc):
    return df[STRAIN].map(lambda s: enc["strain2idx"].get(s, 0)).to_numpy(np.int64)

VARIANTS = ["Peptide-only", "Peptide+StrainID", "Peptide+SpeciesDesc",
            "Peptide+Taxonomy", "Shuffled-Desc (control)"]

def build_inputs(df, variant, enc):
    # Returns (esm, cond, strain_idx) numpy arrays for a variant.
    e = esm_matrix(df)
    n = len(df)
    s_idx = np.zeros(n, np.int64)
    if variant == "Peptide-only":
        cond = np.zeros((n, 0), np.float32)
    elif variant == "Peptide+StrainID":
        cond = np.zeros((n, 0), np.float32); s_idx = strain_idx(df, enc)
    elif variant == "Peptide+SpeciesDesc":
        cond = species_emb_matrix(df)
    elif variant == "Peptide+Taxonomy":
        cond = taxonomy_matrix(df, enc)
    elif variant == "Shuffled-Desc (control)":
        cond = species_emb_matrix(df, SHUFFLE_MAP)
    else:
        raise ValueError(variant)
    return e, cond, s_idx

def _loader(df, variant, enc, shuffle):
    e, cond, s = build_inputs(df, variant, enc)
    y = df[TARGET].to_numpy(np.float32)[:, None]
    ds = TensorDataset(torch.from_numpy(e), torch.from_numpy(cond),
                       torch.from_numpy(s), torch.from_numpy(y))
    return DataLoader(ds, batch_size=BATCH if shuffle else 512, shuffle=shuffle)

def train_variant(variant, tr, va, enc, verbose=False):
    set_seed()
    cond_dim = build_inputs(tr.head(1), variant, enc)[1].shape[1]
    n_strains = len(enc["strain2idx"]) if variant == "Peptide+StrainID" else 0
    model = AblationMLP(cond_dim=cond_dim, n_strains=n_strains).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-5)
    crit = nn.SmoothL1Loss()                      # Huber loss
    tl, vl = _loader(tr, variant, enc, True), _loader(va, variant, enc, False)
    best, best_state, bad = float("inf"), None, 0
    for ep in range(1, EPOCHS + 1):
        model.train()
        for e_, c_, s_, y_ in tl:
            e_, c_, s_, y_ = e_.to(DEVICE), c_.to(DEVICE), s_.to(DEVICE), y_.to(DEVICE)
            opt.zero_grad(); loss = crit(model(e_, c_, s_), y_); loss.backward(); opt.step()
        sched.step()
        model.eval(); vloss = 0.0
        with torch.no_grad():
            for e_, c_, s_, y_ in vl:
                e_, c_, s_, y_ = e_.to(DEVICE), c_.to(DEVICE), s_.to(DEVICE), y_.to(DEVICE)
                vloss += crit(model(e_, c_, s_), y_).item() * len(y_)
        vloss /= len(va)
        if vloss < best - 1e-5:
            best, best_state, bad = vloss, {k: v.cpu().clone() for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break
        if verbose and (ep % 10 == 0 or ep == 1):
            print(f"  [{variant}] epoch {ep:02d} val Huber {vloss:.4f}")
    model.load_state_dict(best_state)
    return model

@torch.no_grad()
def predict(model, df, variant, enc):
    model.eval()
    e, cond, s = build_inputs(df, variant, enc)
    e = torch.from_numpy(e).to(DEVICE); cond = torch.from_numpy(cond).to(DEVICE); s = torch.from_numpy(s).to(DEVICE)
    return model(e, cond, s).cpu().numpy().flatten()

def metrics(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    mae = float(np.mean(np.abs(y - p)))
    mse = float(np.mean((y - p) ** 2))
    rmse = float(np.sqrt(mse))
    ss = float(1 - np.sum((y - p) ** 2) / np.sum((y - y.mean()) ** 2)) if len(y) > 1 and y.std() > 0 else np.nan
    def _safe(fn):
        if len(y) < 3 or y.std() == 0 or p.std() == 0:
            return np.nan
        return float(fn(y, p)[0])
    return {"N": int(len(y)), "MAE": mae, "MSE": mse, "RMSE": rmse, "R2": ss,
            "Pearson": _safe(pearsonr), "Spearman": _safe(spearmanr), "Kendall": _safe(kendalltau)}

print("Model, encoders, training loop and metrics ready. Variants:", VARIANTS)

## 6 — Evaluation A: standard clustered split

Train each variant on the (cluster-disjoint) train split, early-stop on
validation, evaluate once on the held-out test split.

In [ ]:
enc_std = fit_encoders(train_df)
std_models, std_preds, std_rows = {}, {}, []
for v in VARIANTS:
    print("Training:", v)
    m = train_variant(v, train_df, val_df, enc_std)
    p = predict(m, test_df, v, enc_std)
    std_models[v], std_preds[v] = m, p
    row = {"Variant": v, **metrics(test_df[TARGET].values, p)}
    std_rows.append(row)

std_results = pd.DataFrame(std_rows).set_index("Variant").round(4)
print("\n=== Evaluation A — standard clustered test split ===")
display(std_results)
std_results.to_csv(CACHE_DIR / "eval_A_standard.csv")

## 7 — Evaluation B: zero-shot (unseen strains)

Whole **strain** labels are removed from training and used only for test, and any
CD-HIT cluster appearing in the test set is dropped from training so
near-identical peptides cannot straddle the split. This measures cross-strain
generalisation: the *species* may still appear in training under a different
strain, so these are unseen **strains**, not unseen species. Pathogen-side
features (species description / taxonomy) can in principle transfer to a new
strain, whereas a learned Strain-ID cannot.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

pool = pd.concat([train_df, val_df, test_df], ignore_index=True)
counts = pool[STRAIN].value_counts()
eligible = counts[(counts >= 20) & (counts.index != "unknown")].index.to_numpy()
rng_zs = np.random.default_rng(SEED); rng_zs.shuffle(eligible)
n_hold = max(1, int(np.ceil(0.10 * len(eligible))))
heldout = set(eligible[:n_hold])

zs_test = pool[pool[STRAIN].isin(heldout)].reset_index(drop=True)
rest = pool[~pool[STRAIN].isin(heldout)]
# Remove whole CD-HIT clusters present in the test set (not just exact sequences)
# so near-identical peptides cannot straddle the split.
rest = rest[~rest[CLUSTER].isin(set(zs_test[CLUSTER]))].reset_index(drop=True)
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
tr_i, va_i = next(gss.split(rest, groups=rest[CLUSTER].astype(str)))
zs_train, zs_val = rest.iloc[tr_i].reset_index(drop=True), rest.iloc[va_i].reset_index(drop=True)

print(f"Held-out strains: {len(heldout)} | zero-shot test rows: {len(zs_test)} | "
      f"train/val: {len(zs_train)}/{len(zs_val)}")

enc_zs = fit_encoders(zs_train)
zs_rows, zs_preds = [], {}
for v in VARIANTS:
    m = train_variant(v, zs_train, zs_val, enc_zs)
    p = predict(m, zs_test, v, enc_zs); zs_preds[v] = p
    zs_rows.append({"Variant": v, **metrics(zs_test[TARGET].values, p)})

zs_results = pd.DataFrame(zs_rows).set_index("Variant").round(4)
print("\n=== Evaluation B — zero-shot held-out targets ===")
display(zs_results)
zs_results.to_csv(CACHE_DIR / "eval_B_zeroshot.csv")

## 8 — Evaluation C: few-shot (sparse targets)

Targets with only **5–20 training rows**. We score each variant on the standard
test rows that belong to these sparse targets — the regime where borrowing
pathogen-side information should help most.

In [ ]:
train_counts = train_df[STRAIN].value_counts()
few_targets = set(train_counts[(train_counts >= 5) & (train_counts <= 20)].index)
few_mask = test_df[STRAIN].isin(few_targets)
few_test = test_df[few_mask].reset_index(drop=True)
print(f"Few-shot targets (5-20 train rows): {len(few_targets)} | "
      f"test rows on these targets: {len(few_test)}")

if len(few_test) >= 5:
    few_rows = []
    for v in VARIANTS:                              # reuse models trained in Evaluation A
        p = std_preds[v][few_mask.to_numpy()]
        few_rows.append({"Variant": v, **metrics(few_test[TARGET].values, p)})
    few_results = pd.DataFrame(few_rows).set_index("Variant").round(4)
    print("\n=== Evaluation C — few-shot sparse targets ===")
    display(few_results)
    few_results.to_csv(CACHE_DIR / "eval_C_fewshot.csv")
else:
    print("Too few few-shot test rows for a stable evaluation.")
    few_results = pd.DataFrame()

## 9 — Long-tail analysis

Per-target MAE, RMSE, Spearman rho and Kendall tau on the standard test split,
bucketed by training frequency — **Low (<20)**, **Medium (20-100)**,
**High (>100)** — with one summary table and a grouped bar chart across all five
models.

In [ ]:
import matplotlib.pyplot as plt

def bucket(n):
    return "Low (<20)" if n < 20 else ("Medium (20-100)" if n <= 100 else "High (>100)")

train_counts_all = train_df[STRAIN].value_counts().to_dict()
test_df = test_df.copy()
test_df["_bucket"] = test_df[STRAIN].map(lambda s: bucket(train_counts_all.get(s, 0)))

# Per-target metrics (>=5 test rows) for each variant.
per_target = []
for v in VARIANTS:
    tmp = test_df.assign(_pred=std_preds[v])
    for strain, g in tmp.groupby(STRAIN):
        if len(g) < 5:
            continue
        m = metrics(g[TARGET].values, g["_pred"].values)
        per_target.append({"Variant": v, "Target": strain, "N_test": len(g),
                           "Bucket": g["_bucket"].iloc[0],
                           "MAE": m["MAE"], "RMSE": m["RMSE"],
                           "Spearman": m["Spearman"], "Kendall": m["Kendall"]})
per_target_df = pd.DataFrame(per_target)
per_target_df.to_csv(CACHE_DIR / "long_tail_per_target.csv", index=False)

order = ["Low (<20)", "Medium (20-100)", "High (>100)"]
summary = (per_target_df.groupby(["Bucket", "Variant"])[["MAE", "RMSE", "Spearman", "Kendall"]]
           .mean().round(4).reset_index())
summary["Bucket"] = pd.Categorical(summary["Bucket"], order, ordered=True)
summary = summary.sort_values(["Bucket", "Variant"])
print("=== Long-tail summary (mean per-target metric by bucket x model) ===")
display(summary)
summary.to_csv(CACHE_DIR / "long_tail_summary.csv", index=False)
print("Targets per bucket:",
      {b: int((per_target_df.drop_duplicates('Target')['Bucket'] == b).sum()) for b in order})

In [ ]:
# Grouped bar chart: mean per-target MAE by frequency bucket across the 5 models.
pivot = summary.pivot(index="Bucket", columns="Variant", values="MAE").reindex(order)
colors = ["#4C78A8", "#F58518", "#54A24B", "#B279A2", "#9D9D9D"]
x = np.arange(len(order)); w = 0.15
fig, ax = plt.subplots(figsize=(10, 5.5))
for i, v in enumerate(VARIANTS):
    ax.bar(x + (i - 2) * w, pivot[v].values, w, label=v, color=colors[i % len(colors)])
ax.set_xticks(x); ax.set_xticklabels(order)
ax.set_ylabel("Mean per-target MAE (pMIC)  -  lower is better")
ax.set_xlabel("Target frequency in training data")
ax.set_title("Long-tail: per-target MAE by training frequency across 5 feature regimes")
ax.legend(fontsize=8, ncol=2)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(CACHE_DIR / "long_tail_mae.png", dpi=150)
plt.show()

## 10 — Architectural upgrade: FiLM conditioning

Concatenation forces the MLP to learn peptide-pathogen interactions from
scratch. **FiLM** (Feature-wise Linear Modulation) instead lets the pathogen
embedding generate a scale (gamma) and shift (beta) that *directly modulate* the
peptide embedding:

$$h_{\text{mod}} = \gamma(z_{\text{pathogen}}) \odot z_{\text{peptide}} + \beta(z_{\text{pathogen}})$$

Biologically, this lets each pathogen gate *which* peptide features matter — a
Gram-negative LPS outer membrane and a Gram-positive thick peptidoglycan wall
reward different charge/hydrophobicity profiles. We compare FiLM against plain
concatenation using the **same** PubMedBERT species-description signal, under all
three evaluation protocols.

In [ ]:
# FiLM regressor: pathogen (species-description) embedding modulates the peptide
# (ESM-2) embedding. gamma is centred at 1 (1 + gamma) for training stability.
class FiLMRegressor(nn.Module):
    def __init__(self, esm_dim=ESM_DIM, cond_dim=PUBMEDBERT_DIM, proj=128, hidden=HID, dropout=DROPOUT):
        super().__init__()
        self.pep = nn.Linear(esm_dim, proj)
        self.film = nn.Sequential(nn.Linear(cond_dim, proj), nn.GELU(), nn.Linear(proj, 2 * proj))
        self.post = nn.Sequential(nn.LayerNorm(proj), nn.GELU(), nn.Dropout(dropout))
        self.head = nn.Sequential(
            nn.Linear(proj, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, 1))
    def forward(self, x_esm, x_cond):
        h = self.pep(x_esm)
        gamma, beta = self.film(x_cond).chunk(2, dim=-1)
        h = (1.0 + gamma) * h + beta            # FiLM modulation
        return self.head(self.post(h))

def _film_tensors(df):
    return (torch.from_numpy(esm_matrix(df)),
            torch.from_numpy(species_emb_matrix(df)),
            torch.from_numpy(df[TARGET].to_numpy(np.float32)[:, None]))

def train_film(tr, va):
    set_seed()
    model = FiLMRegressor().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-5)
    crit = nn.SmoothL1Loss()
    e, c, y = _film_tensors(tr)
    tl = DataLoader(TensorDataset(e, c, y), batch_size=BATCH, shuffle=True)
    ve, vc, vy = (t.to(DEVICE) for t in _film_tensors(va))
    best, best_state, bad = float("inf"), None, 0
    for ep in range(1, EPOCHS + 1):
        model.train()
        for be, bc, by in tl:
            be, bc, by = be.to(DEVICE), bc.to(DEVICE), by.to(DEVICE)
            opt.zero_grad(); crit(model(be, bc), by).backward(); opt.step()
        sched.step(); model.eval()
        with torch.no_grad():
            vloss = crit(model(ve, vc), vy).item()
        if vloss < best - 1e-5:
            best, best_state, bad = vloss, {k: v.cpu().clone() for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    return model

@torch.no_grad()
def predict_film(model, df):
    model.eval()
    e, c, _ = _film_tensors(df)
    return model(e.to(DEVICE), c.to(DEVICE)).cpu().numpy().flatten()

In [ ]:
# Train FiLM on the standard and zero-shot training sets, then compare FiLM vs
# concatenation for the species-description signal across all three protocols.
film_std = train_film(train_df, val_df)
film_zs  = train_film(zs_train, zs_val)
p_film_std = predict_film(film_std, test_df)
p_film_zs  = predict_film(film_zs, zs_test)

SD = "Peptide+SpeciesDesc"
comp = []
for proto, y, concat_pred, film_pred in [
    ("A Standard",  test_df[TARGET].values, std_preds[SD], p_film_std),
    ("B Zero-shot", zs_test[TARGET].values, zs_preds[SD], p_film_zs),
]:
    comp.append({"Protocol": proto, "Conditioning": "Species-Desc (concat)", **metrics(y, concat_pred)})
    comp.append({"Protocol": proto, "Conditioning": "Species-Desc (FiLM)",   **metrics(y, film_pred)})

if len(few_test) >= 5:
    m = few_mask.to_numpy(); yf = few_test[TARGET].values
    comp.append({"Protocol": "C Few-shot", "Conditioning": "Species-Desc (concat)", **metrics(yf, std_preds[SD][m])})
    comp.append({"Protocol": "C Few-shot", "Conditioning": "Species-Desc (FiLM)",   **metrics(yf, p_film_std[m])})

film_vs_concat = pd.DataFrame(comp).set_index(["Protocol", "Conditioning"]).round(4)
print("=== FiLM vs concatenation (same PubMedBERT species-description signal) ===")
display(film_vs_concat)
film_vs_concat.to_csv(CACHE_DIR / "film_vs_concat.csv")

## 11 — Biological feature enrichment

Three enrichments that PathoMIC-style species/sequence models miss:

- **Terminal modifications** — binary DBAASP flags for C-terminal amidation and
  N-terminal acetylation (amidation alone often shifts MIC 2-4x).
- **Short-peptide biophysics** — net charge at pH 7, hydrophobic moment, Boman
  index, length, concatenated with ESM-2 (which was pretrained on long proteins
  and under-represents short-peptide amphipathicity).
- **Resistance-aware text** — strain-specific resistance traits (MRSA, MDR,
  carbapenem-resistant, efflux overexpression, ...) parsed from the strain label
  and appended to the PubMedBERT description, so resistant vs. susceptible
  strains of the same species get distinct embeddings.

We bundle all three into one **Enriched** model and compare it to Peptide-only
and Species-Description across the three protocols.

In [ ]:
# --- (a) Terminal modification flags from DBAASP metadata ---
MOD_COL = next((c for c in ["api_modifications_json", "chemical_form_signature",
                            "modifications", "api_modifications"] if c in train_df.columns), None)
print("Modification column:", MOD_COL or "NONE (flags default to 0)")

def mod_flags(df):
    n = len(df)
    if MOD_COL is None:
        return np.zeros((n, 2), np.float32)
    txt = df[MOD_COL].fillna("").astype(str).str.lower()
    c_amide = txt.str.contains("amid").to_numpy(np.float32)                 # C-terminal amidation
    n_acetyl = txt.str.contains("acetyl").to_numpy(np.float32)              # N-terminal acetylation
    return np.stack([c_amide, n_acetyl], axis=1).astype(np.float32)

# --- (b) Short-peptide biophysical descriptors ---
EISENBERG = {"A":0.62,"R":-2.53,"N":-0.78,"D":-0.90,"C":0.29,"Q":-0.85,"E":-0.74,"G":0.48,
             "H":-0.40,"I":1.38,"L":1.06,"K":-1.50,"M":0.64,"F":1.19,"P":0.12,"S":-0.18,
             "T":-0.05,"W":0.81,"Y":0.26,"V":1.08}
BOMAN = {"L":-4.92,"I":-4.92,"V":-4.04,"F":-2.98,"M":-2.35,"W":-2.33,"A":-1.81,"C":-1.28,
         "G":-0.94,"Y":0.14,"T":2.57,"S":3.40,"H":4.66,"Q":5.54,"K":5.55,"N":6.64,
         "E":6.81,"D":8.72,"R":14.92,"P":0.0}
POS_PKA = {"K":10.5, "R":12.5, "H":6.0}
NEG_PKA = {"D":3.9, "E":4.1, "C":8.3, "Y":10.1}
NT_PKA, CT_PKA = 9.0, 3.1

def _net_charge(seq, pH=7.0):
    pos = 1.0 / (1.0 + 10 ** (pH - NT_PKA))
    neg = 1.0 / (1.0 + 10 ** (CT_PKA - pH))
    for a in seq:
        if a in POS_PKA:   pos += 1.0 / (1.0 + 10 ** (pH - POS_PKA[a]))
        elif a in NEG_PKA: neg += 1.0 / (1.0 + 10 ** (NEG_PKA[a] - pH))
    return pos - neg

def _hydro_moment(seq, delta=100.0):
    import math
    rad = math.radians(delta)
    sx = sum(EISENBERG.get(a, 0.0) * math.cos(i * rad) for i, a in enumerate(seq))
    sy = sum(EISENBERG.get(a, 0.0) * math.sin(i * rad) for i, a in enumerate(seq))
    return math.sqrt(sx * sx + sy * sy) / max(len(seq), 1)

def _boman(seq):
    return sum(BOMAN.get(a, 0.0) for a in seq) / max(len(seq), 1)

_bio_cache = {}
def _bio(seq):
    seq = str(seq).upper()
    if seq not in _bio_cache:
        _bio_cache[seq] = np.array([_net_charge(seq), _hydro_moment(seq), _boman(seq), float(len(seq))], np.float32)
    return _bio_cache[seq]

_bio_train = np.stack([_bio(s) for s in train_df[SEQ]])
BIO_MU, BIO_SD = _bio_train.mean(0), _bio_train.std(0) + 1e-8      # fixed train-set normalisation
def biophys_matrix(df):
    return ((np.stack([_bio(s) for s in df[SEQ]]) - BIO_MU) / BIO_SD).astype(np.float32)

print("Biophysics = [net_charge_pH7, hydrophobic_moment, boman_index, length]")
_ex = _bio(train_df[SEQ].iloc[0])
print("  example:", dict(zip(["charge", "muH", "boman", "len"], _ex.round(3).tolist())))
_f = mod_flags(train_df)
print("  C-term amidation: %.1f%% of train | N-term acetylation: %.1f%%" % (100 * _f[:, 0].mean(), 100 * _f[:, 1].mean()))

In [ ]:
# --- (c) Resistance-aware strain descriptions -> PubMedBERT (cached) ---
RESIST_RULES = [
    (["mrsa", "methicillin"],                         "methicillin-resistant (MRSA)"),
    (["vre", "vancomycin"],                           "vancomycin-resistant"),
    (["carbapenem", "kpc", "cre", "crab", "ndm", "oxa-48"], "carbapenem-resistant"),
    (["esbl"],                                        "extended-spectrum beta-lactamase producing"),
    (["mdr", "multidrug", "multi-drug", "xdr"],       "multidrug-resistant"),
    (["efflux"],                                      "efflux-pump overexpressing"),
    (["colistin", "mcr-1", "polymyxin"],              "colistin-resistant"),
]

def resistance_clause(strain_label):
    t = str(strain_label).lower()
    traits = [phrase for keys, phrase in RESIST_RULES if any(k in t for k in keys)]
    if traits:
        return " This strain is annotated as " + ", ".join(traits) + "."
    return " No specific antibiotic-resistance phenotype is annotated for this strain."

strain_pairs = (pd.concat([train_df, val_df, test_df])[[STRAIN, SPECIES]]
                .drop_duplicates().reset_index(drop=True))
strain_desc_map = {r[STRAIN]: species_meta.get(r[SPECIES], describe_species(r[SPECIES]))[3] + resistance_clause(r[STRAIN])
                   for _, r in strain_pairs.iterrows()}

STRAIN_PT = CACHE_DIR / "strain_pubmedbert_emb.pt"
if STRAIN_PT.exists():
    blob = torch.load(STRAIN_PT, weights_only=False)      # dict of numpy arrays
    strain_emb_vecs = {k: np.asarray(v, np.float32) for k, v in blob["emb"].items()}
    new = [s for s in strain_desc_map if s not in strain_emb_vecs]
    print(f"Loaded {len(strain_emb_vecs)} cached strain embeddings; {len(new)} new.")
    if new:
        strain_emb_vecs.update(compute_pubmedbert({s: strain_desc_map[s] for s in new}))
        torch.save({"emb": strain_emb_vecs, "desc": strain_desc_map}, STRAIN_PT)
else:
    print(f"Encoding {len(strain_desc_map)} resistance-aware strain descriptions with PubMedBERT.")
    strain_emb_vecs = compute_pubmedbert(strain_desc_map)
    torch.save({"emb": strain_emb_vecs, "desc": strain_desc_map}, STRAIN_PT)
    print("Saved ->", STRAIN_PT)

def strain_desc_emb_matrix(df):
    return np.stack([strain_emb_vecs.get(s, _zero768) for s in df[STRAIN]]).astype(np.float32)

_res = [s for s in strain_desc_map if "annotated as" in strain_desc_map[s]]
print(f"Strains with a parsed resistance phenotype: {len(_res)}")
if _res:
    print("  example:", strain_desc_map[_res[0]])

In [ ]:
# --- Enriched model: ESM-2 + biophysics + mod flags + resistance-aware text ---
def enriched_cond(df):
    return np.hstack([biophys_matrix(df), mod_flags(df), strain_desc_emb_matrix(df)]).astype(np.float32)

def train_concat(tr, va, cond_builder):
    set_seed()
    model = AblationMLP(cond_dim=cond_builder(tr.head(1)).shape[1], n_strains=0).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-5)
    crit = nn.SmoothL1Loss()
    def ten(d):
        return (torch.from_numpy(esm_matrix(d)), torch.from_numpy(cond_builder(d)),
                torch.from_numpy(d[TARGET].to_numpy(np.float32)[:, None]))
    e, c, y = ten(tr)
    tl = DataLoader(TensorDataset(e, c, y), batch_size=BATCH, shuffle=True)
    ve, vc, vy = (t.to(DEVICE) for t in ten(va))
    best, best_state, bad = float("inf"), None, 0
    for ep in range(1, EPOCHS + 1):
        model.train()
        for be, bc, by in tl:
            be, bc, by = be.to(DEVICE), bc.to(DEVICE), by.to(DEVICE)
            opt.zero_grad(); crit(model(be, bc, None), by).backward(); opt.step()
        sched.step(); model.eval()
        with torch.no_grad():
            vloss = crit(model(ve, vc, None), vy).item()
        if vloss < best - 1e-5:
            best, best_state, bad = vloss, {k: v.cpu().clone() for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    return model

@torch.no_grad()
def predict_concat(model, df, cond_builder):
    model.eval()
    e = torch.from_numpy(esm_matrix(df)).to(DEVICE)
    c = torch.from_numpy(cond_builder(df)).to(DEVICE)
    return model(e, c, None).cpu().numpy().flatten()

enr_std = train_concat(train_df, val_df, enriched_cond)
enr_zs  = train_concat(zs_train, zs_val, enriched_cond)
p_enr_std = predict_concat(enr_std, test_df, enriched_cond)
p_enr_zs  = predict_concat(enr_zs, zs_test, enriched_cond)

ENR = "Enriched (bio+mods+resistance)"
rows = []
for proto, y, preds in [
    ("A Standard",  test_df[TARGET].values,
     {"Peptide-only": std_preds["Peptide-only"], "Species-Desc": std_preds["Peptide+SpeciesDesc"], ENR: p_enr_std}),
    ("B Zero-shot", zs_test[TARGET].values,
     {"Peptide-only": zs_preds["Peptide-only"],  "Species-Desc": zs_preds["Peptide+SpeciesDesc"],  ENR: p_enr_zs}),
]:
    for name, p in preds.items():
        rows.append({"Protocol": proto, "Model": name, **metrics(y, p)})
if len(few_test) >= 5:
    m = few_mask.to_numpy(); yf = few_test[TARGET].values
    for name, p in [("Peptide-only", std_preds["Peptide-only"][m]),
                    ("Species-Desc", std_preds["Peptide+SpeciesDesc"][m]),
                    (ENR, p_enr_std[m])]:
        rows.append({"Protocol": "C Few-shot", "Model": name, **metrics(yf, p)})

enriched_results = pd.DataFrame(rows).set_index(["Protocol", "Model"]).round(4)
print("=== Biological feature enrichment: Enriched vs baselines ===")
display(enriched_results)
enriched_results.to_csv(CACHE_DIR / "enriched_results.csv")

## 12 — Best model: Strain-ID + enriched peptide + FiLM

Peptide+StrainID is the strongest few-shot baseline (MSE ~0.23), so we treat it
as the **anchor** and build the best model on top of it:

- **Strain-ID anchor** = learned embedding of the exact strain label.
- **Peptide stream** = ESM-2 embedding + biophysics (4) + modification flags (2),
  concatenated with the strain embedding.
- **Pathogen conditioner** = resistance-aware strain PubMedBERT embedding (768).
- **Fusion** = FiLM — the pathogen gates *which* peptide+strain features matter.

Biophysics and modifications stay on the **peptide** side (only the pathogen
embedding drives γ/β). We benchmark against Peptide+StrainID and the other strong
configurations across all three protocols and on the long tail.

In [ ]:
class FiLMEnriched(nn.Module):
    # Strain-ID anchor (learned embedding) + enriched peptide stream, FiLM-gated by pathogen text.
    def __init__(self, pep_dim, n_strains, strain_dim=32, cond_dim=PUBMEDBERT_DIM, proj=128, hidden=HID, dropout=DROPOUT):
        super().__init__()
        self.strain_emb = nn.Embedding(n_strains, strain_dim)
        self.pep = nn.Linear(pep_dim + strain_dim, proj)
        self.film = nn.Sequential(nn.Linear(cond_dim, proj), nn.GELU(), nn.Linear(proj, 2 * proj))
        self.post = nn.Sequential(nn.LayerNorm(proj), nn.GELU(), nn.Dropout(dropout))
        self.head = nn.Sequential(
            nn.Linear(proj, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, 1))
    def forward(self, x_pep, x_cond, strain_idx):
        h = self.pep(torch.cat([x_pep, self.strain_emb(strain_idx)], dim=-1))
        gamma, beta = self.film(x_cond).chunk(2, dim=-1)
        h = (1.0 + gamma) * h + beta            # pathogen-gated peptide+strain features
        return self.head(self.post(h))

def pep_matrix(df):                              # full peptide stream
    return np.hstack([esm_matrix(df), biophys_matrix(df), mod_flags(df)]).astype(np.float32)

def _ten2(d, pep_b, cond_b, enc):
    return (torch.from_numpy(pep_b(d)), torch.from_numpy(cond_b(d)),
            torch.from_numpy(strain_idx(d, enc)),
            torch.from_numpy(d[TARGET].to_numpy(np.float32)[:, None]))

def train_film2(tr, va, pep_b, cond_b, enc):
    set_seed()
    model = FiLMEnriched(pep_dim=pep_b(tr.head(1)).shape[1], n_strains=len(enc["strain2idx"])).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-5)
    crit = nn.SmoothL1Loss()
    e, c, s, y = _ten2(tr, pep_b, cond_b, enc)
    tl = DataLoader(TensorDataset(e, c, s, y), batch_size=BATCH, shuffle=True)
    ve, vc, vs, vy = (t.to(DEVICE) for t in _ten2(va, pep_b, cond_b, enc))
    best, best_state, bad = float("inf"), None, 0
    for ep in range(1, EPOCHS + 1):
        model.train()
        for be, bc, bs, by in tl:
            be, bc, bs, by = be.to(DEVICE), bc.to(DEVICE), bs.to(DEVICE), by.to(DEVICE)
            opt.zero_grad(); crit(model(be, bc, bs), by).backward(); opt.step()
        sched.step(); model.eval()
        with torch.no_grad():
            vloss = crit(model(ve, vc, vs), vy).item()
        if vloss < best - 1e-5:
            best, best_state, bad = vloss, {k: v.cpu().clone() for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    return model

@torch.no_grad()
def predict_film2(model, df, pep_b, cond_b, enc):
    model.eval()
    e = torch.from_numpy(pep_b(df)).to(DEVICE); c = torch.from_numpy(cond_b(df)).to(DEVICE)
    s = torch.from_numpy(strain_idx(df, enc)).to(DEVICE)
    return model(e, c, s).cpu().numpy().flatten()

best_std = train_film2(train_df, val_df, pep_matrix, strain_desc_emb_matrix, enc_std)
best_zs  = train_film2(zs_train, zs_val, pep_matrix, strain_desc_emb_matrix, enc_zs)
p_best_std = predict_film2(best_std, test_df, pep_matrix, strain_desc_emb_matrix, enc_std)
p_best_zs  = predict_film2(best_zs, zs_test, pep_matrix, strain_desc_emb_matrix, enc_zs)

In [ ]:
# Benchmark the best model against the strongest prior configurations.
# Anchor = Peptide+StrainID (the strongest few-shot baseline), not Peptide-only.
BEST = "Strain-ID+Enriched+FiLM (best)"
ANCHOR = "Peptide+StrainID"
rows = []
for proto, y, preds in [
    ("A Standard",  test_df[TARGET].values,
     {ANCHOR: std_preds[ANCHOR], "Enriched (concat)": p_enr_std,
      "Species-Desc (FiLM)": p_film_std, BEST: p_best_std}),
    ("B Zero-shot", zs_test[TARGET].values,
     {ANCHOR: zs_preds[ANCHOR], "Enriched (concat)": p_enr_zs,
      "Species-Desc (FiLM)": p_film_zs, BEST: p_best_zs}),
]:
    for name, p in preds.items():
        rows.append({"Protocol": proto, "Model": name, **metrics(y, p)})
if len(few_test) >= 5:
    m = few_mask.to_numpy(); yf = few_test[TARGET].values
    for name, p in [(ANCHOR, std_preds[ANCHOR][m]), ("Enriched (concat)", p_enr_std[m]),
                    ("Species-Desc (FiLM)", p_film_std[m]), (BEST, p_best_std[m])]:
        rows.append({"Protocol": "C Few-shot", "Model": name, **metrics(yf, p)})

best_model_results = pd.DataFrame(rows).set_index(["Protocol", "Model"]).round(4)
print("=== Best model vs strongest prior configurations (anchor = Peptide+StrainID) ===")
display(best_model_results)
best_model_results.to_csv(CACHE_DIR / "best_model_results.csv")

# Long-tail: does the best model beat the Strain-ID anchor most on rare targets?
lt = []
for name, preds in [(ANCHOR, std_preds[ANCHOR]), (BEST, p_best_std)]:
    tmp = test_df.assign(_pred=preds)
    for b in order:
        sub = tmp[tmp["_bucket"] == b]
        maes = [float(np.mean(np.abs(g[TARGET].values - g["_pred"].values)))
                for _, g in sub.groupby(STRAIN) if len(g) >= 5]
        lt.append({"Model": name, "Bucket": b,
                   "Mean per-target MAE": round(float(np.mean(maes)), 4) if maes else np.nan,
                   "#targets": len(maes)})
best_long_tail = pd.DataFrame(lt)
print("\n=== Best model vs peptide-only on the long tail (mean per-target MAE) ===")
display(best_long_tail.pivot(index="Bucket", columns="Model", values="Mean per-target MAE").reindex(order))
best_long_tail.to_csv(CACHE_DIR / "best_model_long_tail.csv", index=False)

## 13 — Summary

- **Evaluation A** (`eval_A_standard.csv`) — how much each pathogen-side feature
  adds on the standard cluster-disjoint test.
- **Evaluation B** (`eval_B_zeroshot.csv`) — generalisation to completely unseen
  strains; Strain-ID collapses to the unknown bucket while species description /
  taxonomy can transfer.
- **Evaluation C** (`eval_C_fewshot.csv`) — the sparse-target regime where
  pathogen priors should help most.
- **Long-tail** (`long_tail_summary.csv`, `long_tail_mae.png`) — per-target
  error by training frequency.
- **FiLM vs concat** (`film_vs_concat.csv`) — whether feature-wise modulation of
  the peptide embedding by the pathogen beats plain concatenation of the same
  species-description signal.
- **Enriched model** (`enriched_results.csv`) — ESM-2 + terminal-modification
  flags + short-peptide biophysics + resistance-aware strain text, vs the
  peptide-only and species-description baselines.
- **Best model** (`best_model_results.csv`, `best_model_long_tail.csv`) —
  Strain-ID anchor + enriched peptide stream (ESM-2 + biophysics + mod flags)
  with FiLM gating by the resistance-aware pathogen embedding; benchmarked
  against the Peptide+StrainID anchor, with its long-tail gain quantified per
  frequency bucket.

The **Shuffled-Description control** must not beat the real
Species-Description variant; if it does, the apparent gain is capacity/noise,
not biological signal. All artifacts are cached under `PROJECT_DIR/ablation_cache`.


## 14 — Benchmark comparison vs PathoMIC

PathoMIC (arXiv 2608.26228v1) reports **MSE on log10-MIC**, **Spearman rho** and
**Kendall tau** under few-shot and zero-shot. Our target is
`pMIC = 6 - log10(MIC / uM)`, so squared error and rank correlations are directly
comparable (negation + constant offset leave MSE and rank metrics unchanged).
Our encoder is **ESM2-8M** (their ESM2-8M row is the closest backbone). We show
our **Strain-ID anchor** (ESM2-8M + learned strain embedding — strain-conditioned,
so compare it to their pathogen-aware family, not their seq-only row) and our best
model.

**Not a same-benchmark result.** Protocols and data differ: PathoMIC splits
per *species* (8:1:1; few-shot = species with <20 train records, zero-shot =
species with <10 records held out) over 48,429 records / 366 species; we split
by CD-HIT cluster and hold out *strains* (few-shot = strains with 5-20 train
rows). Treat this as an approximate reference, not a head-to-head.

In [ ]:
# PathoMIC reference values (arXiv 2608.26228v1, Tables 1-2; log10-MIC, mean of 3 runs).
PATHOMIC = [
    ("Few-shot",  "PathoMIC", "ESM2-8M (seq-only)",      0.2493, 0.7327, 0.5532),
    ("Few-shot",  "PathoMIC", "ESM2-150M (seq-only)",    0.2479, 0.7329, 0.5536),
    ("Few-shot",  "PathoMIC", "ESM2-150M-SP (best)",     0.1528, 0.8474, 0.6784),
    ("Zero-shot", "PathoMIC", "ESM2-8M (seq-only)",      0.4338, 0.6021, 0.4378),
    ("Zero-shot", "PathoMIC", "ESM2-150M (seq-only)",    0.4220, 0.6115, 0.4446),
    ("Zero-shot", "PathoMIC", "ESM2-150M-GNN (best)",    0.4118, 0.6305, 0.4636),
]

def _our(proto, model):
    try:
        r = best_model_results.loc[(proto, model)]
        return round(float(r["MSE"]), 4), round(float(r["Spearman"]), 4), round(float(r["Kendall"]), 4)
    except Exception:
        return None

rows = []
for setting, proto in [("Few-shot", "C Few-shot"), ("Zero-shot", "B Zero-shot")]:
    for s, src, model, mse, sp, kt in PATHOMIC:
        if s == setting:
            rows.append({"Setting": setting, "Source": src, "Model": model,
                         "MSE": mse, "Spearman": sp, "Kendall": kt})
    for model, label in [("Peptide+StrainID", f"ESM2-{ESM_BACKBONE} + Strain-ID"),
                         ("Strain-ID+Enriched+FiLM (best)", f"ESM2-{ESM_BACKBONE} Strain-ID+Enriched+FiLM (best)")]:
        g = _our(proto, model)
        if g:
            rows.append({"Setting": setting, "Source": "Ours", "Model": label,
                         "MSE": g[0], "Spearman": g[1], "Kendall": g[2]})

pathomic_comparison = pd.DataFrame(rows)[["Setting", "Source", "Model", "MSE", "Spearman", "Kendall"]]
print("=== Ours vs PathoMIC (MSE lower is better; Spearman/Kendall higher is better) ===")
print("Approximate reference only: different dataset and split protocol (see notes above).")
display(pathomic_comparison)
pathomic_comparison.to_csv(CACHE_DIR / "pathomic_comparison.csv", index=False)